#### 1. Environment & Directory Setup

In [1]:
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta

folders = ['../data/raw', '../data/processed', '../app']
for folder in folders:
    os.makedirs(folder, exist_ok=True)

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Environment and Folder Structure Verified.")

Environment and Folder Structure Verified.


#### 2. Defining the Specialized Catalog

In [2]:
# Product Categories & Items
categories = {
    'Pumps': ['Centrifugal Pump', 'Submersible Pump', 'Solar Pump', 'Booster Pump'],
    'Solar': ['330W Solar Panel', '5kVA Hybrid Inverter', '200Ah Gel Battery', 'Solar Controller'],
    'Generators': ['10kVA Diesel Gen', '3kVA Petrol Gen', '50kVA Silent Gen'],
    'Water Treatment': ['UV Sterilizer', 'RO Membrane', 'Chlorine Dosing Pump', 'FRP Vessel']
}

sources = ['Europe', 'Japan', 'Australia', 'Local']
product_list = []

for cat, items in categories.items():
    for item in items:
        source = random.choice(sources)
        
        lead_time = random.randint(3, 10) if source == 'Local' else random.randint(45, 120)
            
        product_list.append({
            'SKU': f"{cat[:3].upper()}-{random.randint(100, 999)}",
            'Product_Name': item,
            'Category': cat,
            'Source': source,
            'Unit_Cost_USD': round(random.uniform(100, 8000), 2),
            'Lead_Time_Days': lead_time,
            'Initial_Stock': random.randint(10, 60)
        })

df_products = pd.DataFrame(product_list)
print(f"Created {len(df_products)} specialized SKUs.")
df_products.head()

Created 15 specialized SKUs.


,SKU,Product_Name,Category,Source,Unit_Cost_USD,Lead_Time_Days,Initial_Stock
0,PUM-859,Centrifugal Pump,Pumps,Europe,2272.73,48,24
1,PUM-792,Submersible Pump,Pumps,Japan,5951.28,58,44
2,PUM-532,Solar Pump,Pumps,Europe,351.08,120,15
3,PUM-617,Booster Pump,Pumps,Japan,4855.95,74,45
4,SOL-529,330W Solar Panel,Solar,Japan,1841.48,114,47


### 3. Simulating 365 Days of Transactional History

In [3]:
# Date range for 1 year
date_range = pd.date_range(start=(datetime.now() - timedelta(days=365)), end=datetime.now())
history_data = []

for _, product in df_products.iterrows():
    # Different categories have different 'velocities'
    # Generators sell slowly (expensive), Water Treatment parts sell faster (consumables)
    lambda_p = 0.05 if product['Category'] == 'Generators' else 0.8
    
    daily_sales = np.random.poisson(lambda_p, len(date_range))
    
    # Add seasonality (High demand for pumps/solar in Ethiopian Dry Season: Oct - May)
    for i, date in enumerate(date_range):
        qty = daily_sales[i]
        if date.month in [10, 11, 12, 1, 2, 3, 4, 5]:
            qty = int(qty * 1.4) # 40% boost in dry season
        
        if qty > 0:
            history_data.append({'Date': date, 'SKU': product['SKU'], 'Quantity_Sold': qty})

df_history = pd.DataFrame(history_data)
print(f"Generated {len(df_history)} historical sales records.")

Generated 2453 historical sales records.


#### 4. Engineering Inventory Metrics

In [4]:
# Aggregate daily demand stats
stats = df_history.groupby('SKU')['Quantity_Sold'].agg(['mean', 'std']).reset_index()
stats.columns = ['SKU', 'Avg_Daily_Demand', 'Demand_Volatility']

# Merge stats with product metadata
df_final = pd.merge(df_products, stats, on='SKU', how='left').fillna(0)

# Calculate Daily Consumption Value (Importance factor)
df_final['Daily_Consumption_USD'] = df_final['Avg_Daily_Demand'] * df_final['Unit_Cost_USD']

print("Inventory Metrics calculated successfully.")
df_final.head()

Inventory Metrics calculated successfully.


,SKU,Product_Name,Category,Source,Unit_Cost_USD,Lead_Time_Days,Initial_Stock,Avg_Daily_Demand,Demand_Volatility,Daily_Consumption_USD
0,PUM-859,Centrifugal Pump,Pumps,Europe,2272.73,48,24,1.590674,0.948308,3615.171554
1,PUM-792,Submersible Pump,Pumps,Japan,5951.28,58,44,1.464789,0.882177,8717.367887
2,PUM-532,Solar Pump,Pumps,Europe,351.08,120,15,1.500000,0.856913,526.620000
3,PUM-617,Booster Pump,Pumps,Japan,4855.95,74,45,1.466667,0.807461,7122.060000
4,SOL-529,330W Solar Panel,Solar,Japan,1841.48,114,47,1.559783,0.973255,2872.308478


#### 5. Data Persistence

In [5]:
raw_path = '../data/raw/simulated_inventory_base.csv'
df_final.to_csv(raw_path, index=False)

print(f"Success! Data saved to: {raw_path}")

Success! Data saved to: ../data/raw/simulated_inventory_base.csv
